In [ ]:
# Runtime check: install missing packages if needed
import sys, subprocess

def ensure_packages(pkgs):
    for p in pkgs:
        try:
            __import__(p)
        except ModuleNotFoundError:
            print(f"Installing {p}...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])

# Ensure sqlalchemy and mysql connector for DB export
ensure_packages(['sqlalchemy', 'mysql-connector-python'])
print('Package install check complete. Restart kernel if needed.')


# Sales Analytics Pipeline
This notebook demonstrates a simple end-to-end pipeline: load CSV, clean data, derive columns, export cleaned CSV, and show how to push data to MySQL for SQL analysis.

## 1. Imports
Required libraries: pandas, numpy, sqlalchemy (for MySQL export). Install with `pip install pandas sqlalchemy mysql-connector-python` if needed.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
print('pandas', pd.__version__)

## 2. Load dataset
Set `DATA_PATH` to your CSV. The example below points to the `Week-1/Shopping_DataSet/Combined_dataset.csv` file in this workspace — update if your file is elsewhere.

In [ ]:
DATA_PATH = '../Week-1/Shopping_DataSet/Combined_dataset.csv'  # adjust as needed
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Quick inspection
Check shape, dtypes, nulls, and duplicates.

In [ ]:
df.shape, df.dtypes
df.isnull().sum()
df.duplicated().sum()

## 4. Cleaning & derived columns
Common steps: drop exact duplicates, fill or drop missing values, convert date columns, and create derived metrics like `TotalRevenue` and `ProfitMargin`.

In [ ]:
# Example cleaning pipeline (adapt column names as needed)
df = df.copy()
# Drop exact duplicates
df = df.drop_duplicates().reset_index(drop=True)
# Example: standardize column names to lowercase
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
# Parse a date column if present
for col in ['date', 'order_date', 'purchase_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        break
# Fill or drop simple missing values (example)
if 'quantity' in df.columns:
if 'price' in df.columns:
# Derived: total revenue per row
if 'quantity' in df.columns and 'price' in df.columns:
# Derived: profit margin if cost column exists
if 'cost' in df.columns and 'price' in df.columns:
df.head()

## 5. Export cleaned CSV
Save a cleaned version for downstream use.

In [ ]:
OUT_PATH = 'cleaned_sales.csv'
df.to_csv(OUT_PATH, index=False)
print('Wrote', OUT_PATH)

## 6. (Optional) Push to MySQL and run sample SQL
Below is an example snippet to write the cleaned dataframe to a MySQL table. Provide your DB credentials securely — do not hardcode secrets in notebooks.

In [ ]:
# Example: write to MySQL (uncomment and configure)
# from sqlalchemy.engine.url import URL
# mysql_url = URL.create(
#     drivername='mysql+mysqlconnector',
#     username='your_user', password='your_password',
#     host='localhost', port=3306, database='sales_db'
# )
# engine = create_engine(mysql_url)
# df.to_sql('cleaned_sales', engine, if_exists='replace', index=False)
# pd.read_sql_query('SELECT * FROM cleaned_sales LIMIT 5', engine)

## 7. Example SQL analysis ideas
- Monthly revenue time series
- Top-selling products and customers
- Region-wise sales and customer lifetime value (requires customer id / region fields)

---
Notebook created by the Sales Analytics Pipeline template. Update paths and credentials before running any DB operations.